# **Legal Research Assistant for UAE Labor Law**

## Import Libraries

In [1]:
!pip install -q faiss-cpu sentence-transformers openai


In [2]:
pip install python-dotenv

In [3]:
import json
import os
from pathlib import Path
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv
import time

## Auto-generate the knowledge base with an LLM

In [4]:
BASE_DIR = Path("/content")
DATA_DIR = BASE_DIR / "data"
INDEX_DIR = BASE_DIR / "index"
ENV_DIR = BASE_DIR / "env"
DATA_DIR.mkdir(exist_ok=True)
INDEX_DIR.mkdir(exist_ok=True)
ENV_DIR.mkdir(exist_ok=True)

In [19]:
LAW_KB_PATH = DATA_DIR / "law_knowledge_base_fixed.json"
with open(LAW_KB_PATH, "r", encoding="utf-8") as f:
    law_kb = json.load(f)

## Build the FAISS retrieval index

In [21]:
EMBEDDING_MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"

In [23]:
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)
embeddings = embedder.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [24]:
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))
faiss.write_index(index, str(INDEX_DIR / "law.index"))
with open(INDEX_DIR / "law_metadata.json", "w", encoding="utf-8") as f:
    json.dump(law_kb, f, ensure_ascii=False, indent=2)

## Setting up an API key

In [25]:
ENV_FILE = ENV_DIR / ".env"
base_url = "https://api.gapapi.com/v1"
model_name = "gpt-4o"

In [26]:
load_dotenv(ENV_FILE)
api_key = os.getenv("API_KEY")
os.environ["LLM_API_KEY"] = api_key
llm_client = OpenAI(api_key=api_key, base_url=base_url)

## Legal Research Using RAG



In [27]:
SIMILARITY_THRESHOLD = 0.35
TOP_K = 3

In [28]:
SYSTEM_PROMPT = (
    "أنت مساعد بحث قانوني تقني (وليس محامياً) موجّه لمحامين محترفين. "
    "أجب على سؤال المحامي باستخدام المعلومات الموجودة في السياق القانوني "
    "المقدم لك فقط. لا تخترع أي رقم أو حكم غير موجود في السياق. اذكر رقم "
    "المادة المرجعية إن وُجدت في السياق. إذا كان السياق لا يغطي السؤال "
    "بوضوح، قل صراحة إن هذا خارج نطاق القاعدة المعرفية المحدودة لهذه الأداة "
    "وينبغي الرجوع للنص الرسمي الكامل. أجب بالعربية الفصحى، بدقة واختصار."
)

In [29]:
def bot_say(text, delay=3):
    tex = f"LRABot: {text}"
    if delay:
        time.sleep(delay)
    return tex

In [30]:
def legal_search(question):
    query_vec = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_vec.astype(np.float32), TOP_K)
    if scores[0][0] < SIMILARITY_THRESHOLD:
        print("  لم يتم العثور على موضوع ذي صلة واضحة في القاعدة المعرفية المحدودة لهذه الأداة.")
        time.sleep(5)
        print("    يرجى الرجوع مباشرة إلى uaelegislation.gov.ae أو استشارة المصدر الرسمي.")
        return
    retrieved = [law_kb[idx] for idx in indices[0] if idx != -1]
    context_block = "\n\n".join(
        f"[{r['topic']} -- {r['article_ref']}]\n{r['text']}" for r in retrieved
    )
    user_prompt = f"سؤال المحامي: {question}\n\nالسياق القانوني:\n{context_block}"
    response = llm_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
    )
    print('LRABot:')
    print()
    time.sleep(3)
    print(" الإجابة:\n")
    time.sleep(3)
    print(response.choices[0].message.content)
    print("\n المصادر المسترجَعة:")
    for r in retrieved:
        print(f"  - {r['topic']} ({r['article_ref']})")
        print(f"    المصدر: {r['source_name']}")
        print(f"    الرابط: {r['source_url']}")

# Run Chatbot

In [31]:
sentenses = [
    "مرحباً أيها المستخدم الكريم",
    "مساعد البحث القانوني جاهز",
    "اكتب سؤالك أدناه"
]

In [32]:
input_str = "سؤالك: "

In [33]:
for sen in sentenses:
  print(bot_say(sen, delay=3))

while True:
    print(bot_say('يرجى كتابة سؤالك في المربع أدناه. شكرًا لك', delay=3))
    time.sleep(3)
    q = input('You:').strip()
    print()
    if q.lower() in {'خروج'}:
        break
    if not q:
        continue
    legal_search(q)
    print()

LRABot: مرحباً أيها المستخدم الكريم
LRABot: مساعد البحث القانوني جاهز
LRABot: اكتب سؤالك أدناه
LRABot: يرجى كتابة سؤالك في المربع أدناه. شكرًا لك
You:ما هي حالات إنهاء عقد العمل المشروعة؟

LRABot:

 الإجابة:

حالات إنهاء عقد العمل المشروعة وفقاً للمادة 42 من المرسوم بقانون اتحادي رقم 33 لسنة 2021 تشمل:

1. اتفاق الطرفين كتابياً على إنهاء العقد.
2. انتهاء مدة العقد دون تمديد أو تجديد.
3. إرادة أحد الطرفين مع الالتزام بالإشعار.
4. وفاة أحد الطرفين.
5. ظروف قاهرة تمنع استمرار العقد.
6. الإفلاس أو الإعسار أو وجود أسباب اقتصادية أو استثنائية تمنع استمرار المشروع.
7. إغلاق المنشأة بصورة نهائية وفق التشريعات والإجراءات المعمول بها.

هذه الحالات تتطلب الالتزام بالشروط والإجراءات المحددة في اللائحة التنفيذية والتشريعات النافذة.

 المصادر المسترجَعة:
  - إنهاء العقد لأسباب اقتصادية أو استثنائية (المادة 42 -- Federal Decree-Law No. 33 of 2021)
    المصدر: Federal Decree-Law No. 33 of 2021
    الرابط: https://u.ae/-/media/Documents-2023/Federal-DecreeLaw-No-33-of-2021-Regarding-the-Regulation-of-E